# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (as a single object)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# Display additional metadata info
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {metadata.keywords}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The Croissant schema organizes entities by their unique `@id`. Here, we'll list the available record sets and their field `@id`s for further exploration.

In [ ]:
# List the record sets defined in the dataset metadata

# The list of record sets maybe on metadata.recordSet. If it is empty, mlcroissant auto-parses from schema:
record_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []

if not record_sets:
    # Use dataset.list_record_sets() if recordSet is not populated
    record_set_ids = dataset.list_record_sets()
else:
    # Otherwise, extract the @id from each recordSet object
    record_set_ids = [rs['@id'] if isinstance(rs, dict) else rs for rs in record_sets]

print("Available Record Sets:")
for rs_id in record_set_ids:
    print(f" - {rs_id}")

# List fields for each record set by their @id
for rs_id in record_set_ids:
    print(f"\nFields in Record Set {rs_id}:")
    fields = dataset.list_fields(record_set=rs_id)
    for f_id in fields:
        print(f"   - {f_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract all record sets, referencing by their `@id`. If the record set contains survey responses, you'll see fields like age, gender, GAD-7 score, PHQ-9 score, etc. Extracting data via `mlcroissant` ensures all column references respect the schema.

In [ ]:
# Load all record sets into DataFrames
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

    print(f"RecordSet {rs_id} loaded.")
    if len(records) > 0:
        print(f"Columns (@id): {df.columns.tolist()}")
        print(df.head())
    else:
        print("No records available.")

# For further analysis, let's pick the main survey record set containing participant responses.
main_record_set = record_set_ids[0] if record_set_ids else None

if main_record_set:
    main_df = dataframes.get(main_record_set, pd.DataFrame())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will reference fields by their `@id`. For example, suppose the survey contains columns with `@id`s like `age`, `gender`, `gad7_score`, `phq9_score`, and `psq_score`.

In [ ]:
# Example: Filter participants with GAD-7 score above threshold and normalize it

# List columns in main_df (they should be the field @id's)
if not main_df.empty:
    print("Columns (@id):", main_df.columns.tolist())

    # Select numeric fields -- replace with real @id's found in previous step
    numeric_field_id = None
    for col in main_df.columns:
        if 'gad' in col.lower():
            numeric_field_id = col
            break
    if numeric_field_id is None:
        for col in main_df.columns:
            if 'phq' in col.lower() or 'psq' in col.lower():
                numeric_field_id = col
                break

    # Select a grouping field (again, use @id)
    group_field_id = None
    for col in main_df.columns:
        if 'gender' in col.lower():
            group_field_id = col
            break

    if numeric_field_id:
        threshold = 10
        # Ensure values are numeric
        filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce') > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric_field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - filtered_df[numeric_field_id].astype(float).mean()
        ) / filtered_df[numeric_field_id].astype(float).std()

        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by gender (if present)
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric score field found for EDA.")
else:
    print("Main participant DataFrame cannot be found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example visualization of GAD-7 scores by gender. Field references are always via their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not main_df.empty and numeric_field_id and group_field_id:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
else:
    print("Cannot generate visualization: missing numeric or group field.")

## 6. Conclusion
In this notebook, we explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using `mlcroissant`. We loaded the dataset, examined its structure via `@id` references, extracted survey data, performed exploratory analysis on numeric mental health scores, normalized values, grouped by demographic variables, and visualized score distributions.

**Key findings:**
- The dataset is organized by Croissant schema, enabling robust AI-ready analysis workflows.
- Records and fields are referenced consistently by their `@id`.
- Exploratory analysis shows variation in mental health indicator scores, and demographic grouping reveals patterns in the data.

Further analysis can be conducted using other fields and record sets as needed.